In [7]:
# Notes on something certain to tackle
# (1) malfunction code: 500, -500, 999, -999
# (2) missing value tokens: blank, ., --, NaN, null, NA, N/A, #N/A, n/a, missing
# (3) convert into datetime64 -> should be strictly increasing, but could have absent months.

In [8]:
import pandas as pd

In [9]:
df_raw = pd.read_csv("global_temp_dirty_v2.csv")

In [10]:
df_raw.head()

,Date,Temperature_Anomaly,Source/Notes
0,188001,-0.73,NaN
1,Feb-1880,-0.752,NaN
2,Feb-1880,-0.752,NaN
3,1880/03,-0.670,NaN
4,1880.04,-0.593,NaN


In [11]:
df_raw.tail()

,Date,Temperature_Anomaly,Source/Notes
1706,2025/12,1.004,NaN
1707,NaN,NaN,NaN
1708,END OF DATA,NaN,NaN
1709,NaN,NaN,NaN
1710,Source: SIMULATED teaching dataset (not offici...,NaN,NaN


In [12]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1711 entries, 0 to 1710
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Date                  1708 non-null   object
 1    Temperature_Anomaly  1673 non-null   object
 2   Source/Notes          77 non-null     object
dtypes: object(3)
memory usage: 40.2+ KB


In [13]:
df_raw.describe()

,Date,Temperature_Anomaly,Source/Notes
count,1708,1673,77
unique,1702,1126,5
top,Feb-1880,500,sensor_B
freq,2,35,19


In [14]:
df_raw["Source/Notes"].unique()

array([nan, 'NOAA', 'sensor_B', 'note: calibration', 'HadCRUT',
       'sensor_A'], dtype=object)

In [15]:
# 3.1

In [22]:
stripped_columns = [col.strip() for col in df_raw.columns]
df_raw.columns = stripped_columns

In [23]:
# Strip the header names
df_raw = df_raw[["Date", "Temperature_Anomaly"]]

In [26]:
mask_end_of_data = df_raw["Date"] == "END OF DATA"

In [34]:
idx_eod_of_data = df_raw[mask_end_of_data].index[0]

In [39]:
df_raw = df_raw.iloc[:idx_eod_of_data, :]

In [55]:
# both columns are null
df_raw = df_raw.loc[~pd.isna(df_raw).all(axis=1)]

In [56]:
num_total = df_raw.shape[0]
print(num_total)

1707


In [64]:
# Repair rows with swapped fields
# attempt to parse value column into datetime
pd.to_datetime(df_raw["Temperature_Anomaly"], errors="coerce").dropna()
# So need to process all years larger than the last record, minus 100

/tmp/ipykernel_1236/1328197056.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(df_raw["Temperature_Anomaly"], errors="coerce").dropna()


,Temperature_Anomaly
157,1893-07-01
182,1898-06-01
224,1901-10-01
275,1903-09-01
296,1906-06-01
307,1907-07-01
332,1908-09-01
371,1911-11-01
430,1916-11-01
653,2036-02-01


In [67]:
df_raw.tail(10)

,Date,Temperature_Anomaly
1697,NaT,1.027
1698,NaT,1.122
1699,NaT,0.949
1700,2025-06-01,1.026
1701,2025-07-01,1.061
1702,NaT,0.999
1703,NaT,.
1704,NaT,1.016
1705,NaT,0.985
1706,2025-12-01,1.004


In [70]:
import datetime as dt
import numpy as np
def try_parse_datetime(val, max_year: int = 2025) -> np.datetime64:
  dt_obj = pd.to_datetime(val, errors="coerce")
  if pd.isna(dt_obj):
    return None
  if dt_obj.year > max_year and dt_obj.year < 2100:
    return dt_obj.replace(year=dt_obj.year - 100).to_datetime64()
  return dt_obj.to_datetime64()

In [72]:
# repair rows with swapped fields, again

In [76]:
mask_swapped_fields = df_raw["Temperature_Anomaly"].apply(lambda x: try_parse_datetime(x, max_year=2026)).dropna().index

In [77]:
actual_date_vals = df_raw.loc[mask_swapped_fields, "Temperature_Anomaly"]
actual_value_vals = df_raw.loc[mask_swapped_fields, "Date"]
df_raw.loc[mask_swapped_fields, "Date"] = actual_date_vals
df_raw.loc[mask_swapped_fields, "Temperature_Anomaly"] = actual_value_vals

In [79]:
# Convert every date to a datetime64 value, set to the first day of its month
df_raw["Date Cleaned"] = df_raw["Date"].apply(lambda x: try_parse_datetime(x, max_year=2026))

/tmp/ipykernel_1236/3353437012.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_raw["Date Cleaned"] = df_raw["Date"].apply(lambda x: try_parse_datetime(x, max_year=2026))


In [84]:
# Date Cleaned: Null Value, Not Increasingn, Duplicate
error_log = []

for i in range(df_raw.shape[0]):
  if i == 0:
    if df_raw.loc[0, "Date Cleaned"] == None:
      error_log.append({
          "Index": i,
          "Error": "Null Value"
      })
  else:
    current_row = df_raw.loc[i, :]
    previous_row = df_raw.loc[i - 1, :]
    if pd.isna(current_row["Date Cleaned"]):
      error_log.append({
          "Index": i,
          "Error": "Null Value"
      })
    elif not pd.isna(previous_row["Date Cleaned"]) and current_row["Date Cleaned"] == previous_row["Date Cleaned"]:
      error_log.append({
          "Index": i,
          "Error": "Duplicate"
      })
    elif not pd.isna(previous_row["Date Cleaned"]) and current_row["Date Cleaned"] < previous_row["Date Cleaned"]:
      error_log.append({
          "Index": i,
          "Error": "Not Increasing"
      })


In [91]:
pd.DataFrame(error_log).groupby("Error").count()

,Index
Error,
Duplicate,7
Not Increasing,257
Null Value,214


In [93]:
df_error_log = pd.DataFrame(error_log)


In [99]:
df_raw.loc[df_error_log[df_error_log["Error"] == "Not Increasing"]["Index"]]

,Date,Temperature_Anomaly,Date Cleaned
173,1897-01-01,500,1897-01-01
174,1896-09-01,-0.61,1896-09-01
178,1894-12-01,-0.418,1894-12-01
180,1895-06-01,-0.523°C,1895-06-01
183,1895-03-01,-0.650,1895-03-01
...,...,...,...
1407,1997-12-01,0.632,1997-12-01
1409,1997-07-01,0.68,1997-07-01
1411,1998-09-01,0.667,1998-09-01
1420,1997-05-01,0.592,1997-05-01


In [101]:
# Check duplicate rows
df_raw.loc[df_error_log[df_error_log["Error"] == "Duplicate"]["Index"]-1]

,Date,Temperature_Anomaly,Date Cleaned
1,1880-02-01,-0.752,1880-02-01
40,1883-05-01,-0.630,1883-05-01
335,1908-12-01,"-0,601",1908-12-01
413,1915-06-01,-0.261,1915-06-01
618,2033-04-01,-0.261,1933-04-01
1050,2069-12-01,0.220,1969-12-01
1098,2074-01-01,0.231,1974-01-01


In [102]:
df_raw.loc[df_error_log[df_error_log["Error"] == "Duplicate"]["Index"]]

,Date,Temperature_Anomaly,Date Cleaned
2,1880-02-01,-0.752,1880-02-01
41,1883-05-01,-0.630,1883-05-01
336,1908-12-01,"-0,601",1908-12-01
414,1915-06-01,-0.261,1915-06-01
619,2033-04-01,-0.261,1933-04-01
1051,2069-12-01,0.220,1969-12-01
1099,1974-01-01,0.231,1974-01-01


In [107]:
df_error_log[df_error_log["Error"].isin(["Duplicate", "Not Increasing"])]["Index"]

,Index
0,2
3,41
6,173
7,174
8,178
...,...
328,1407
329,1409
330,1411
334,1420
